# Scoring and replacement value
This notebook demonstrates which calculations are configured facts and which are transparent heuristics. It uses only labeled synthetic projections and does not train a production model.

In [ ]:
import pandas as pd

from fantasy_draft_ai.rules.models import DraftSettings, FlexSlot, LeagueRules
from fantasy_draft_ai.rules.replacement import position_demand, replacement_levels
from fantasy_draft_ai.scoring.engine import PlayerStatLine, ScoringRules, score_player

In [ ]:
line = PlayerStatLine(position="WR", receptions=7, receiving_yards=100, receiving_tds=1)
for label, reception_value in [("standard", 0), ("half PPR", 0.5), ("PPR", 1)]:
    print(label, score_player(line, ScoringRules(reception=reception_value)))

The same football stat line changes value because scoring is a league rule, not because the football projection changed.

In [ ]:
rules = LeagueRules(
    season=2026,
    teams=12,
    draft=DraftSettings(rounds=16),
    starters={"QB": 1, "RB": 2, "WR": 3, "TE": 1},
    flex_slots=(FlexSlot(name="FLEX", count=2, eligible=("RB", "WR", "TE")),),
    bench=7,
    scoring=ScoringRules(reception=1),
)
position_demand(rules, "WR")

In [ ]:
rows = []
for position, offset in [("QB", 60), ("RB", 30), ("WR", 25), ("TE", 0)]:
    for rank in range(100):
        rows.append(
            {
                "player_id": f"{position}_{rank:03d}",
                "position": position,
                "projected_points": 300 + offset - 2 * rank,
            }
        )
synthetic_projections = pd.DataFrame(rows)
levels = replacement_levels(synthetic_projections, rules)
pd.DataFrame([level.__dict__ for level in levels.values()])

Try changing WR from 3 to 2 or FLEX from 2 to 1. The last-starter boundary changes because league demand changed. This is a documented heuristic; learned player projections and draft availability will arrive in later phases.